In [0]:
from pyspark.sql import functions as F
events = spark.read.table("default.bronze_events")

In [0]:
display(events)

In [0]:
events.printSchema()

In [0]:
from pyspark.sql import functions as F

interaction_df = events.withColumn(
    "rating",
    F.when(F.col("event_type") == "purchase", 3)
     .when(F.col("event_type") == "cart", 2)
     .otherwise(1)
).select("user_id", "product_id", "rating")

In [0]:
interaction_df.show(5)
interaction_df.describe().show()

In [0]:
interaction_df = interaction_df.groupBy(
    "user_id", "product_id"
).agg(
    F.sum("rating").alias("rating")
)

In [0]:
interaction_df.show(5)

In [0]:
sample_df = interaction_df.sample(0.3, seed=42)

In [0]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="user_id",
    itemCol="product_id",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True,
    rank=5,        # reduce
    maxIter=5,     # reduce
    regParam=0.1
)

als_model = als.fit(sample_df)

In [0]:
predictions = als_model.transform(interaction_df)
predictions.show(5)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("user_id").orderBy(F.desc("prediction"))

top5 = predictions.withColumn(
    "rank",
    row_number().over(window)
).filter(F.col("rank") <= 5)

top5.show(5)